# Configuração

## Imports

In [1]:
from pathlib import Path
import shutil
import random
from collections import Counter
from sklearn.model_selection import train_test_split

## PATHS

In [ ]:
#imagens
APSPC1 = Path("APSPC1/APSPC1_unzip/APSPC1")
APSPC2 = Path("APSPC2/APSPC2_unzip/APSPC2")
SEM_DEFEITOS = Path("guandong_dataset/guangdong_round1_train2_20180916/amostras_sem_defeito")

#labels
LABELS_T = Path("paic_workspace/dataset_yolo/train/labels")
LABELS_V = Path("paic_workspace/dataset_yolo/val/labels")

#novo dataset
OUTPUT = Path("dataset_APDDD_completo")

## Constantes

In [3]:
#proporções
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

#seed da divisão
SEED = 42

#classes
CLASSES = [
    "sink-mark",
    "impact-damage",
    "scratch",
    "paint-bubble",
    "orange-peel",
    "bottom-leakage",
    "corner-bottom-leakage",
    "dirty-spots",
    "non-conductive",
    "variegated-color",
]

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

## Funções auxiliares

In [4]:
def find_images(folder):
    """Encontra todas as imagens dentro de uma pasta, inclusive subpastas."""
    return [
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

In [5]:
def load_labels():
    """
    Carrega todas as labels do dataset antigo.
    Retorna:
        dict: {nome_do_arquivo_sem_extensao: caminho_da_label}
    """
    labels = {}
    for label_dir in [LABELS_T, LABELS_V]:
        if not label_dir.exists():
            print(f"[AVISO] Pasta de labels não encontrada: {label_dir}")
            continue

        for label_path in label_dir.rglob("*.txt"):
            stem = label_path.stem

            if stem in labels:
                raise RuntimeError(
                    f"Label duplicada encontrada para '{stem}':\n"
                    f"  {labels[stem]}\n"
                    f"  {label_path}"
                )

            labels[stem] = label_path
    return labels

In [6]:
def read_classes_from_label(label_path):
    """
    Retorna o conjunto de classes presentes em uma imagem.
    """
    classes = set()
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            class_id = int(line.split()[0])
            classes.add(class_id)
    return classes

In [7]:
def stratified_split(items, train_ratio, val_ratio, test_ratio, seed):

    positives = [
        item for item in items
        if not item["negative"]
    ]

    negatives = [
        item for item in items
        if item["negative"]
    ]

    # POSITIVAS
    positive_train, positive_temp = train_test_split(
        positives,
        test_size=(val_ratio + test_ratio),
        random_state=seed,
        shuffle=True
    )

    relative_test_size = test_ratio / (val_ratio + test_ratio)

    positive_val, positive_test = train_test_split(
        positive_temp,
        test_size=relative_test_size,
        random_state=seed,
        shuffle=True
    )

    # NEGATIVAS
    negative_train, negative_temp = train_test_split(
        negatives,
        test_size=(val_ratio + test_ratio),
        random_state=seed,
        shuffle=True
    )

    negative_val, negative_test = train_test_split(
        negative_temp,
        test_size=relative_test_size,
        random_state=seed,
        shuffle=True
    )

    #JUNTA POSITIVAS + NEGATIVAS

    splits = {
        "train": positive_train + negative_train,
        "val": positive_val + negative_val,
        "test": positive_test + negative_test,
    }

    # Embaralha cada split
    rng = random.Random(seed)

    for split in splits:
        rng.shuffle(splits[split])

    return splits

In [8]:
def copy_item(item, split, source_name):
    """
    Copia uma imagem e sua label para o novo dataset.

    source_name é utilizado no nome do arquivo para evitar colisões
    entre APSPC1 e APSPC2.
    """
    image = item["image"]

    # Prefixo evita colisão de nomes entre datasets
    new_stem = f"{source_name}_{image.stem}"
    
    destination_image = (
        OUTPUT / "images" / split / f"{new_stem}{image.suffix.lower()}"
    )
    destination_label = (
        OUTPUT / "labels" / split / f"{new_stem}.txt"
    )

    shutil.copy2(image, destination_image)
    if item["negative"]:
        # Cria arquivo TXT vazio
        destination_label.touch()
    else:
        shutil.copy2(item["label"], destination_label)

In [9]:
def create_yaml():
    """
    Cria o data.yaml.
    """
    yaml_path = OUTPUT / "data.yaml"

    # Caminho absoluto para facilitar o uso com Ultralytics
    dataset_path = OUTPUT.resolve()

    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write(f"path: '{dataset_path}'\n")
        f.write("train: images/train\n")
        f.write("val: images/val\n")
        f.write("test: images/test\n\n")

        f.write(f"nc: {len(CLASSES)}\n")
        f.write("names:\n")

        for i, name in enumerate(CLASSES):
            f.write(f"  {i}: {name}\n")
    return yaml_path

In [10]:
def print_statistics(splits):
    """
    Exibe estatísticas do dataset.
    """
    print("\n" + "=" * 70)
    print("ESTATÍSTICAS DO DATASET")
    print("=" * 70)

    for split, items in splits.items():
        positives = [
            x for x in items
            if not x["negative"]
        ]
        negatives = [
            x for x in items
            if x["negative"]
        ]
        
        class_counter = Counter()
        for item in positives:
            for cls in item["classes"]:
                class_counter[cls] += 1

        print(f"\n{split.upper()}")
        print("-" * 40)
        print(f"Total de imagens : {len(items)}")
        print(f"Com defeito     : {len(positives)}")
        print(f"Sem defeito     : {len(negatives)}")

        print("\nImagens por classe:")
        for i, name in enumerate(CLASSES):
            print(
                f"  {i:2d} - {name:<25} "
                f"{class_counter[i]}"
            )

# Execução

In [11]:
for folder in [
    APSPC1,
    APSPC2,
    SEM_DEFEITOS,
    LABELS_T,
    LABELS_V,
]:

    if not folder.exists():
        raise FileNotFoundError(
            f"Pasta não encontrada:\n{folder.resolve()}"
        )

## Controle do dataset

In [12]:
def exists():
    if OUTPUT.exists():
    
        answer = input(
            f"\nA pasta '{OUTPUT}' já existe.\n"
            "Deseja apagá-la e recriar? [s/N]: "
        )
    
        if answer.lower() != "s":
            print("Operação cancelada.")
            return
    
        shutil.rmtree(OUTPUT)
    
    for split in ["train", "val", "test"]:
    
        (OUTPUT / "images" / split).mkdir(
            parents=True,
            exist_ok=True
        )
    
        (OUTPUT / "labels" / split).mkdir(
            parents=True,
            exist_ok=True
        )

exists()

## Carregando labels

In [13]:
labels = load_labels()
print(f"Labels encontradas: {len(labels)}")

Labels encontradas: 1885


## Carregando imagens com defeito

In [14]:
images_apspc1 = find_images(APSPC1)
print(f"APSPC1: {len(images_apspc1)} imagens")

images_apspc2 = find_images(APSPC2)
print(f"APSPC2: {len(images_apspc2)} imagens")

APSPC1: 940 imagens
APSPC2: 945 imagens


## Carregando imagens sem defeito

In [15]:
images_negative = find_images(SEM_DEFEITOS)
print(f"Sem defeito: {len(images_negative)} imagens")

Sem defeito: 1018 imagens


## Lista de imagens positivas

In [16]:
items = []
missing_labels = []

for source_name, images in [
    ("APSPC1", images_apspc1),
    ("APSPC2", images_apspc2),
]:
    for image in images:
        stem = image.stem
        
        if stem not in labels:
            missing_labels.append(image)
            continue

        label = labels[stem]
        classes = read_classes_from_label(label)
        items.append({
            "image": image,
            "label": label,
            "classes": classes,
            "negative": False,
            "source": source_name,
        })


In [17]:
#verificação se está faltando label
if missing_labels:
    print("\n[ERRO] Foram encontradas imagens sem label:")

    for image in missing_labels[:20]:
        print(f"  {image}")
    if len(missing_labels) > 20:
        print(
            f"  ... e mais "
            f"{len(missing_labels) - 20}"
        )

    raise RuntimeError(
        f"{len(missing_labels)} imagens positivas "
        "não possuem labels correspondentes."
    )

## Adicionando imagens negativas

In [18]:
for image in images_negative:
    items.append({
        "image": image,
        "label": None,
        "classes": set(),
        "negative": True,
        "source": "SEM_DEFEITO",
    })

## Verificando proporção do dataset

In [19]:
positive_count = sum(
    not x["negative"]
    for x in items
)
negative_count = sum(
    x["negative"]
    for x in items
)

print(f"Imagens com defeito : {positive_count}")
print(f"Imagens sem defeito : {negative_count}")
print(f"Total               : {len(items)}")

print(
    f"Proporção positivas : "
    f"{positive_count / len(items) * 100:.2f}%"
)
print(
    f"Proporção negativas : "
    f"{negative_count / len(items) * 100:.2f}%"
)

Imagens com defeito : 1885
Imagens sem defeito : 1018
Total               : 2903
Proporção positivas : 64.93%
Proporção negativas : 35.07%


## Divisão

In [20]:
splits = stratified_split(
    items,
    TRAIN_RATIO,
    VAL_RATIO,
    TEST_RATIO,
    SEED
)

In [21]:
for split, split_items in splits.items():
    print(
        f"  {split}: "
        f"{len(split_items)} imagens"
    )

    for item in split_items:
        copy_item(
            item,
            split,
            item["source"]
        )

  train: 2031 imagens
  val: 436 imagens
  test: 436 imagens


## Criando o YAML

In [22]:
yaml_path = create_yaml()

print(f"\nYAML:")
print(yaml_path.resolve())


YAML:
/home/aline/aline_paic2025/dataset_APDDD/dataset_APDDD_completo/data.yaml


## Estatísticas da divisão

In [23]:
print_statistics(splits)


ESTATÍSTICAS DO DATASET

TRAIN
----------------------------------------
Total de imagens : 2031
Com defeito     : 1319
Sem defeito     : 712

Imagens por classe:
   0 - sink-mark                 64
   1 - impact-damage             32
   2 - scratch                   123
   3 - paint-bubble              210
   4 - orange-peel               162
   5 - bottom-leakage            69
   6 - corner-bottom-leakage     78
   7 - dirty-spots               144
   8 - non-conductive            577
   9 - variegated-color          116

VAL
----------------------------------------
Total de imagens : 436
Com defeito     : 283
Sem defeito     : 153

Imagens por classe:
   0 - sink-mark                 15
   1 - impact-damage             6
   2 - scratch                   30
   3 - paint-bubble              49
   4 - orange-peel               28
   5 - bottom-leakage            14
   6 - corner-bottom-leakage     17
   7 - dirty-spots               28
   8 - non-conductive            135
   9 - varieg